# Fase 3 — Ingeniería de variables y partición agrupada

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Este notebook cierra el diseño antes de ajustar cualquier modelo. Aquí dejo cuatro cosas:

1. **La población de análisis** con la etiqueta v1.1. La operación confirmó que los créditos 131 y 250 fueron incumplimientos, así que entran como eventos: 733 créditos y 23 eventos.
2. **Los predictores**: un núcleo de 11 entradas elegidas por dominio, con su signo esperado, y las variables que quedan fuera con su razón. Incluye la decisión sobre `recaudo` y sobre los contadores de mora del buró.
3. **La información previa de Home Credit** que usará el modelo M3.
4. **Las particiones** de la validación cruzada repetida, agrupadas por cliente, y las pruebas automáticas que demuestran que el diseño no tiene fugas.

La justificación de cada decisión está en `00_protocolo/enmienda_v1.1.md`, que quedó registrada en git antes de este notebook.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import hashlib
import json
import subprocess
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

from tesis_v2 import config, contexto, diseno, estadistica as est, figuras, io_datos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 90)
figuras.estilo()
config.asegurar_carpetas()

D, ctx, ds = contexto.cargar()
print(f"Población primaria (etiqueta v1.1): {len(D)} créditos, {int(D['y'].sum())} eventos, "
      f"{D['grupo_cliente'].nunique()} clientes")
assert (len(D), int(D["y"].sum())) == (733, 23)

Población primaria (etiqueta v1.1): 733 créditos, 23 eventos, 357 clientes


## 1. Poblaciones de análisis

Cada análisis de sensibilidad del protocolo usa una población distinta. Las construyo todas con la misma función (`diseno.poblacion`) para que la única diferencia sea la que el análisis quiere estudiar.

In [2]:
casos = [
    ("Principal: 90+/12m, etiqueta v1.1", dict()),
    ("Etiqueta original de la Fase 1", dict(etiqueta="original")),
    ("60+/12m", dict(objetivo="y_60_12m")),
    ("30+/6m", dict(objetivo="y_30_6m")),
    ("90+/12m, variante B", dict(objetivo="y_90_12m_B")),
    ("Indeterminados = no evento", dict(indeterminados="no_evento")),
    ("Indeterminados = evento", dict(indeterminados="evento")),
    ("Sin la cohorte 2022", dict(excluir_cohortes=(2021, 2022))),
]
filas = []
for nombre, kw in casos:
    P = diseno.poblacion(ds, **kw)
    lo, hi = est.wilson(P["y"].sum(), len(P))
    filas.append({"poblacion": nombre, "creditos": len(P), "eventos": int(P["y"].sum()),
                  "clientes": P["grupo_cliente"].nunique(), "tasa_%": round(100 * P["y"].mean(), 2),
                  "ic95_%": f"{100 * lo:.1f}–{100 * hi:.1f}"})
S = ctx.supervivencia
filas.append({"poblacion": "Supervivencia (libranza 2022+, censurados incluidos)", "creditos": len(S),
              "eventos": int(S["evento_90"].sum()), "clientes": S["grupo_cliente"].nunique(), "tasa_%": None, "ic95_%": "—"})
pobl = pd.DataFrame(filas)
io_datos.guardar_resultado(pobl, "fase3", "poblaciones")
pobl

,poblacion,creditos,eventos,clientes,tasa_%,ic95_%
0,"Principal: 90+/12m, etiqueta v1.1",733,23,357,3.14,2.1–4.7
1,Etiqueta original de la Fase 1,731,21,356,2.87,1.9–4.4
2,60+/12m,733,29,357,3.96,2.8–5.6
3,30+/6m,790,64,372,8.10,6.4–10.2
4,"90+/12m, variante B",733,126,357,17.19,14.6–20.1
5,Indeterminados = no evento,773,23,369,2.98,2.0–4.4
6,Indeterminados = evento,773,63,369,8.15,6.4–10.3
7,Sin la cohorte 2022,684,17,345,2.49,1.6–3.9
8,"Supervivencia (libranza 2022+, censurados incluidos)",1083,29,574,NaN,—


**Lectura.** La reclasificación de 131 y 250 sube la tasa primaria de 2,87 % a 3,14 %. Es poco en términos absolutos, pero son dos eventos más sobre 21, casi un 10 % más de información sobre el desenlace que más importa.

Las poblaciones de sensibilidad tienen dos papeles:

- **Más eventos.** 30+/6m (64 eventos) y la cota con indeterminados como evento (63) dan más potencia, a costa de cambiar lo que se predice.
- **Mismo objetivo, con el peor caso de la etiqueta.** La variante B infla mucho la tasa porque extiende la observación de los cierres sospechosos hasta el corte.

## 2. Predictores del núcleo

Elegí las variables antes de mirar su desempeño en validación. Usé tres criterios: que se conozcan en t0, que existan en todas las cohortes y que tengan una hipótesis de dominio sobre su efecto. Las transformaciones de esta tabla son deterministas, fila a fila: un logaritmo no aprende nada de los datos. Las que sí aprenden (mediana para imputar, estandarización, cuantiles y WoE) se ajustan dentro de cada pliegue con `Pipeline`.

In [3]:
X = diseno.construir_X(D)
descr = {
    "log_monto": "log del valor total del crédito", "log_plazo": "log del plazo según el plan (meses)",
    "edad_t0": "edad en t0 (años); mediana del pliegue si falta", "tiene_historia": "tuvo al menos un crédito previo",
    "log_n_previos": "log(1 + créditos previos)", "meses_cliente": "meses desde el primer crédito (0 si es nuevo)",
    "log_peor_mora_previa": "log(1 + peor DPD previo hasta t0 − 1 día; 0 si es nuevo)",
    "previo_adverso": "último cierre con saldo o por refinanciación", "previo_favorable": "último cierre a término o prepago",
    "n_activos_t0": "créditos previos activos en t0", "log_emp_previos": "log(1 + créditos previos del empleador)"}
signo = {1: "+", -1: "−", 0: "?"}
nucleo = pd.DataFrame({"variable": diseno.NUCLEO, "construccion": [descr[v] for v in diseno.NUCLEO],
                       "signo_esperado": [signo[diseno.SIGNO_ESPERADO[v]] for v in diseno.NUCLEO],
                       "en_home_credit": [v in diseno.COMPARTIDAS for v in diseno.NUCLEO],
                       "%_faltante": X.isna().mean().mul(100).round(1).to_numpy(),
                       "media": X.mean().round(3).to_numpy(), "de": X.std().round(3).to_numpy()})
io_datos.guardar_resultado(nucleo, "fase3", "predictores_nucleo")
nucleo

,variable,construccion,signo_esperado,en_home_credit,%_faltante,media,de
0,log_monto,log del valor total del crédito,?,False,0.0,14.953,1.082
1,log_plazo,log del plazo según el plan (meses),+,True,0.0,2.652,0.742
2,edad_t0,edad en t0 (años); mediana del pliegue si falta,−,True,0.4,39.306,11.001
3,tiene_historia,tuvo al menos un crédito previo,−,True,0.0,0.531,0.499
4,log_n_previos,log(1 + créditos previos),−,True,0.0,0.541,0.575
5,meses_cliente,meses desde el primer crédito (0 si es nuevo),−,True,0.0,7.864,10.183
6,log_peor_mora_previa,log(1 + peor DPD previo hasta t0 − 1 día; 0 si es nuevo),+,True,0.0,0.994,1.391
7,previo_adverso,último cierre con saldo o por refinanciación,+,False,0.0,0.195,0.397
8,previo_favorable,último cierre a término o prepago,−,False,0.0,0.171,0.376
9,n_activos_t0,créditos previos activos en t0,+,False,0.0,0.379,0.518


In [4]:
corr = X.corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.6, 5.4))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=70, fontsize=7)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=7)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=5.5,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlación de Spearman entre predictores del núcleo")
figuras.guardar(fig, "fase3", "F3_01", "correlacion_nucleo",
                "Las variables de historia están muy correlacionadas entre sí; la penalización reparte el efecto en lugar de elegir una al azar.")

**Lectura.** Las cinco variables de historia (`tiene_historia`, `log_n_previos`, `meses_cliente`, `log_peor_mora_previa` y `n_activos_t0`) están fuertemente correlacionadas. Es esperable: todas valen 0 en un cliente nuevo. No elimino ninguna a mano. Cada una tiene una hipótesis propia, y la penalización L2 o el *prior* reparten el efecto común de forma estable. La selección automática por p-valor, en cambio, escogería una al azar en cada muestra. El monto y el plazo tienen una correlación moderada; la experiencia del empleador es casi independiente del resto.

**Eventos por parámetro.** Con 23 eventos y 11 predictores hay **2,1 eventos por parámetro**, muy por debajo de la regla tradicional de 10 a 20. El cálculo de Riley de la Fase 1 ya lo anticipaba (0,7 a 2 parámetros sin contracción). Por eso ningún modelo de esta tesis se estima sin penalización, sin *prior* o sin restricciones de forma.

## 3. Variables que no entran y por qué

Dos variables merecen una explicación propia, porque ni Eider ni la operación pudieron precisar su significado.

- **`recaudo` (modalidad de recaudo).** En la Fase 2 se comportó como un indicador de época, no de riesgo:
  - el recaudo directo solo aparece hasta 2023Q3;
  - su PSI frente a 2023 es el más alto de todas las variables (7,7);
  - dentro de 2023 su relación con el evento se invierte.

  Un predictor que no tiene una definición operativa no se puede mantener en producción: nadie sabría cuándo un cambio en su distribución es un cambio de riesgo y cuándo un cambio de registro. Lo más defendible es **excluirlo del modelo principal**. Lo uso solo en una sensibilidad: si al agregarlo cambian el *log loss* o los coeficientes del núcleo, lo leo como evidencia de confusión con el tiempo, no como una señal aprovechable.
- **`veces_30mas_12m` y `veces_90mas_12m` (buró).** Tienen dos problemas independientes, y cualquiera de los dos bastaría para excluirlas:
  - su semántica no está verificada: no sé si cuentan obligaciones, meses o eventos;
  - tienen **0 % de disponibilidad** en los créditos de desarrollo y 7,6 % en los elegibles 2025+.

  Tampoco las conecto con Home Credit a través del *prior*: transferir un efecto a una variable cuyo significado no conozco sería dar una falsa precisión. Quedan para el monitoreo de la Fase 6, cuando la operación documente la definición y haya créditos maduros con buró.

In [5]:
excluidas = pd.DataFrame([
    ("recaudo", "Significado no documentado; indicador de época (PSI 7,7; relación invertida dentro de 2023)", "Solo sensibilidad"),
    ("veces_30mas_12m, veces_90mas_12m", "Semántica no verificada y 0 % disponible en desarrollo", "Excluidas (Fase 6)"),
    ("resto del buró y PILA", "0 % disponible en desarrollo; 7,6 % en 2025+", "Excluidas"),
    ("tiene_aval, aval_pct, cobra_seguro, cobra_estudio, cobra_4x1000", "Parámetros de política que cambian de golpe (PSI 2,7–7)", "Excluidas"),
    ("tipo_solicitud, es_refinanciacion", "La marca de refinanciación solo existe desde 2024Q2", "Excluidas"),
    ("tasa_mensual", "Precio fijado según el perfil: endógena al riesgo percibido", "Excluida"),
    ("cuota_mensual, n_cuotas", "Redundantes con monto y plazo", "Excluidas"),
    ("monto_sobre_solicitado, con_prenda, sin_empleador", "10 % disponible / 0,4 % / constante", "Excluidas"),
    ("género", "Decisión de Eider: eliminado por completo; no hay auditoría de equidad por género", "Excluida"),
    ("identidad, LAFT/SARLAFT, geografía, huella digital", "Por principio (AGENTS.md, minimización de datos)", "Excluidas"),
], columns=["variables", "razon", "tratamiento"])
io_datos.guardar_resultado(excluidas, "fase3", "variables_excluidas")
excluidas

,variables,razon,tratamiento
0,recaudo,"Significado no documentado; indicador de época (PSI 7,7; relación invertida dentro de ...",Solo sensibilidad
1,"veces_30mas_12m, veces_90mas_12m",Semántica no verificada y 0 % disponible en desarrollo,Excluidas (Fase 6)
2,resto del buró y PILA,"0 % disponible en desarrollo; 7,6 % en 2025+",Excluidas
3,"tiene_aval, aval_pct, cobra_seguro, cobra_estudio, cobra_4x1000","Parámetros de política que cambian de golpe (PSI 2,7–7)",Excluidas
4,"tipo_solicitud, es_refinanciacion",La marca de refinanciación solo existe desde 2024Q2,Excluidas
5,tasa_mensual,Precio fijado según el perfil: endógena al riesgo percibido,Excluida
6,"cuota_mensual, n_cuotas",Redundantes con monto y plazo,Excluidas
7,"monto_sobre_solicitado, con_prenda, sin_empleador","10 % disponible / 0,4 % / constante",Excluidas
8,género,Decisión de Eider: eliminado por completo; no hay auditoría de equidad por género,Excluida
9,"identidad, LAFT/SARLAFT, geografía, huella digital","Por principio (AGENTS.md, minimización de datos)",Excluidas


## 4. Home Credit como información previa

En la v1, Home Credit entraba como datos apilados con el 80 % del peso. La Fase 2 mostró que es otra población: un clasificador las distingue con AUC 0,96. Lo que sí puede transferirse es la **dirección y la magnitud aproximada** de efectos generales del riesgo de consumo: edad, plazo e historia crediticia.

Ajusto una logística sin penalizar en las 224.647 solicitudes de Home Credit, con las seis variables compartidas construidas igual que en FINTECH. Su coeficiente por unidad natural, multiplicado por la desviación estándar de la variable en el entrenamiento de FINTECH, es el **centro del prior** de M3. La escala τ, elegida en el pliegue interno, decide cuánto se le cree.

In [6]:
hc = io_datos.leer_parquet(config.HOME_CREDIT_MAPEADO, ["edad_t0", "plazo_meses", "n_creditos_previos",
                                                        "antiguedad_cliente_meses", "peor_mora_previa", "y"])
Xh = diseno.construir_X_home_credit(hc)
res = sm.Logit(hc["y"].to_numpy(), sm.add_constant(Xh)).fit(disp=0)
de_fin = X.std()
prior = pd.DataFrame({"variable": diseno.COMPARTIDAS, "beta_hc_por_unidad": res.params[diseno.COMPARTIDAS].to_numpy(),
                      "ic95_hc": [f"{a:.4f} a {b:.4f}" for a, b in res.conf_int().loc[diseno.COMPARTIDAS].to_numpy()],
                      "de_fintech": de_fin[diseno.COMPARTIDAS].to_numpy()})
prior["centro_prior_estandarizado"] = prior["beta_hc_por_unidad"] * prior["de_fintech"]
prior["odds_ratio_por_de"] = np.exp(prior["centro_prior_estandarizado"])
prior["signo_esperado"] = [signo[diseno.SIGNO_ESPERADO[v]] for v in diseno.COMPARTIDAS]
prior["coincide_signo"] = np.sign(prior["beta_hc_por_unidad"]) == [diseno.SIGNO_ESPERADO[v] for v in diseno.COMPARTIDAS]
io_datos.guardar_resultado(prior.round(4), "fase3", "prior_home_credit")
assert np.allclose(prior["beta_hc_por_unidad"], ctx.beta_hc[diseno.COMPARTIDAS], rtol=1e-3)
prior.round(4)

,variable,beta_hc_por_unidad,ic95_hc,de_fintech,centro_prior_estandarizado,odds_ratio_por_de,signo_esperado,coincide_signo
0,edad_t0,-0.0211,-0.0226 a -0.0196,11.0009,-0.2320,0.7929,−,True
1,log_plazo,-0.1190,-0.1554 a -0.0825,0.7415,-0.0882,0.9156,+,False
2,tiene_historia,0.7053,0.6201 a 0.7906,0.4994,0.3522,1.4223,−,False
3,log_n_previos,-0.1434,-0.1821 a -0.1047,0.5750,-0.0825,0.9209,−,True
4,meses_cliente,-0.0095,-0.0102 a -0.0088,10.1835,-0.0968,0.9077,−,True
5,log_peor_mora_previa,0.1895,0.1790 a 0.1999,1.3905,0.2635,1.3014,+,True


**Lectura.** Home Credit confirma la dirección esperada en cuatro de las seis variables: más edad, más créditos previos y más antigüedad como cliente bajan el riesgo, y más mora previa lo sube (OR 1,30 por desviación estándar). Contradice la hipótesis en dos:

- **Tener historia aumenta el riesgo** (OR 1,42). En Home Credit, "créditos previos" son solicitudes anteriores en la entidad, y quien vuelve a pedir es con frecuencia quien necesita liquidez. El efecto neto de un cliente recurrente con pocos créditos queda cerca de cero, porque se compensa con el coeficiente negativo del número de créditos.
- **El plazo largo se asocia con menos riesgo** (OR 0,92). Probablemente refleja que la entidad concede plazos largos a los mejores perfiles.

Todos los centros del prior son modestos: entre −0,23 y 0,35 en la escala estandarizada.

Esto ilustra por qué el *prior* necesita una escala ajustable y no un peso fijo. Si τ es pequeño, M3 impone a FINTECH estos efectos, incluidos los que pueden no aplicar a la libranza colombiana. Si los datos propios los contradicen, el pliegue interno elegirá un τ grande y el modelo se acercará a M2. La comparación M3 frente a M2 es, entonces, una prueba directa de si la información externa ayuda.

## 5. Particiones de la validación cruzada repetida

Uso `StratifiedGroupKFold` de 5 pliegues por grupo de cliente, repetido 20 veces con semillas 20261115 + r. Todos los modelos usan exactamente las mismas particiones, así que sus diferencias se pueden comparar de forma pareada. La asignación por crédito se guarda en `_privado/` porque lleva identificadores; aquí solo muestro el resumen.

In [7]:
F = diseno.particiones(D["grupo_cliente"], D["y"], n_rep=20)
huella = hashlib.sha256(F.tobytes()).hexdigest()[:16]
(config.PRIVADO / "particiones.json").write_text(json.dumps({
    "semilla": config.SEMILLA, "repeticiones": 20, "pliegues": 5, "huella": huella,
    "cid": D["cid"].astype(int).tolist(), "pliegue_por_repeticion": F.tolist()}), encoding="utf-8")
F2 = diseno.particiones(D["grupo_cliente"], D["y"], n_rep=20)
assert (F == F2).all(), "Las particiones no son reproducibles"
for r in range(F.shape[0]):
    assert (pd.Series(F[r]).groupby(D["grupo_cliente"].to_numpy()).nunique() == 1).all()
rp = diseno.resumen_particiones(F, D["y"], D["grupo_cliente"])
io_datos.guardar_resultado(rp, "fase3", "resumen_particiones")
resumen = rp[["creditos", "eventos", "clientes"]].agg(["min", "max"])
print(f"Huella de las particiones (SHA-256, 16 caracteres): {huella}")
print("Verificado: reproducibles por semilla y ningún cliente en dos pliegues de una misma repetición.")
resumen

Huella de las particiones (SHA-256, 16 caracteres): f64cf8b90b908fc1
Verificado: reproducibles por semilla y ningún cliente en dos pliegues de una misma repetición.


,creditos,eventos,clientes
min,146,4,70
max,147,5,73


Cada pliegue de prueba tiene entre 4 y 5 eventos. Es muy poco para evaluar un pliegue por sí solo, y por eso las métricas se calculan sobre las predicciones fuera de pliegue de los 733 créditos en cada repetición, no pliegue a pliegue. La estratificación reparte los eventos de forma pareja y la repetición promedia la variabilidad de la partición.

## 6. Pruebas automáticas anti-fuga

Las pruebas de `tests/test_diseno.py` usan datos sintéticos con la misma estructura del dataset, para que se puedan compartir sin datos de clientes. Verifican cinco cosas:

1. Ningún cliente cae en dos pliegues.
2. Las particiones se reproducen con la misma semilla.
3. Alterar las filas de prueba no cambia los predictores de entrenamiento.
4. Las predicciones de M1, M2, M3 y M5 no cambian si se alteran el desenlace o los predictores del pliegue de prueba.
5. El WoE y los modelos que usan filas extra (supervivencia y objetivos auxiliares) nunca ven a los clientes de prueba.

In [8]:
salida = subprocess.run([sys.executable, "-m", "pytest", "../tests", "-q", "-p", "no:cacheprovider"],
                        capture_output=True, text=True)
print(salida.stdout.strip().split("\n")[-1])
assert salida.returncode == 0, salida.stdout

10 passed in 4.51s


In [9]:
_ = io_datos.escribir_registro("08_variables_y_particion", {
    "poblacion_primaria": {"creditos": len(D), "eventos": int(D["y"].sum()), "clientes": int(D["grupo_cliente"].nunique())},
    "predictores": diseno.NUCLEO, "huella_particiones": huella, "pruebas_anti_fuga": salida.stdout.strip().split("\n")[-1]})
figuras.escribir_catalogo("fase3", "Figuras de la Fase 3")

PosixPath('resultados/fase3/catalogo_figuras.md')

## Conclusiones

- La población primaria queda en **733 créditos, 23 eventos y 357 clientes**, con 131 y 250 reclasificados por confirmación de la operación. La etiqueta original (731/21) pasa a ser sensibilidad.
- El núcleo tiene **11 predictores** elegidos por dominio y disponibles en todas las cohortes. Con 2,1 eventos por parámetro, todo modelo necesita penalización, *prior* o restricciones de forma.
- `recaudo` y los contadores de mora del buró quedan **fuera del modelo principal** por falta de definición operativa (y, en el buró, por falta de datos). `recaudo` se examina solo como sensibilidad.
- Home Credit aporta un *prior* coherente con lo esperado en edad y mora previa, y contrario en historia y plazo. La escala τ decidirá cuánto pesa.
- Las 20 repeticiones de 5 pliegues son reproducibles y respetan el agrupamiento por cliente. Las pruebas anti-fuga pasan.